# IPL Jersey Detection — Improved Pipeline
### Key improvements over v1:
- Richer HSV features (H+S+V histograms, not just H)
- Spatial color grid (4 quadrants per cell) for better localization
- Larger BoVW vocabulary (150 words) built on more images
- `class_weight='balanced'` on all classifiers
- Calibrated SVM via `CalibratedClassifierCV` + `LinearSVC` (fast + accurate)
- Undersample class 0 to reduce imbalance
- Smarter confidence thresholding

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('merged_predictions.csv')
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (2449, 68)


,Image File Name,Train Or Test,c01,c02,c03,c04,c05,c06,c07,c08,...,c57,c58,c59,c60,c61,c62,c63,c64,Verified,Annotator
0,5366_f4b12835.jpg,Test,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,True,E
1,5367_3f6b14d2.jpg,Test,0,0,0,0,0,0,0,0,...,0,0,4,4,4,0,0,0,True,E
2,5368_4df8419d.jpg,Test,0,0,0,0,0,0,0,0,...,0,4,4,0,0,0,0,0,True,E
3,5369_bf9d2878.jpg,Test,0,0,0,0,3,3,0,0,...,0,0,4,3,3,3,3,0,True,E
4,5370_15dbc64e.jpg,Test,0,0,0,0,0,0,0,0,...,4,4,0,0,3,0,0,0,True,E


In [2]:
np.random.seed(42)
train_size = int(0.7 * len(df))
shuffled_indices = np.random.permutation(len(df))
df['Train Or Test'] = 'Test'
df.loc[shuffled_indices[:train_size], 'Train Or Test'] = 'Train'

print('Distribution of Train vs Test:')
print(df['Train Or Test'].value_counts(normalize=True) * 100)

Distribution of Train vs Test:
Train Or Test
Train    69.98775
Test     30.01225
Name: proportion, dtype: float64


## STEP A: Build BoVW Vocabulary (Larger + Better)

In [3]:
import os
import cv2
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from tqdm.notebook import tqdm

print('--- STEP A: BUILDING BAG OF VISUAL WORDS (ORB + K-MEANS) ---')

IMAGE_DIR = 'ipl_2023_master_dataset/ipl2023'
VOCAB_SIZE = 150  # Increased from 50 -> more discriminative visual words

orb = cv2.ORB_create(nfeatures=500)
all_descriptors = []

# Use MORE images for vocabulary — 500 instead of 200
sample_df = df.sample(n=min(500, len(df)), random_state=42)

print('Extracting ORB descriptors to build vocabulary...')
for index, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    img_name = row['Image File Name']
    img_path = None
    for root, dirs, files in os.walk(IMAGE_DIR):
        if img_name in files:
            img_path = os.path.join(root, img_name)
            break
    if img_path is None or not os.path.exists(img_path):
        continue
    img = cv2.imread(img_path)
    if img is None:
        continue
    img = cv2.resize(img, (800, 600))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    keypoints, descriptors = orb.detectAndCompute(gray, None)
    if descriptors is not None:
        all_descriptors.extend(descriptors)

all_descriptors = np.array(all_descriptors, dtype=np.float32)
print(f'Total visual fragments collected: {len(all_descriptors)}')

print(f'Clustering into {VOCAB_SIZE} visual words...')
kmeans_bovw = MiniBatchKMeans(n_clusters=VOCAB_SIZE, batch_size=2000, random_state=42, n_init=3)
kmeans_bovw.fit(all_descriptors)
print('Visual Vocabulary built successfully!')

--- STEP A: BUILDING BAG OF VISUAL WORDS (ORB + K-MEANS) ---
Extracting ORB descriptors to build vocabulary...


  0%|          | 0/500 [00:00<?, ?it/s]

Total visual fragments collected: 250000
Clustering into 150 visual words...
Visual Vocabulary built successfully!


## STEP B: Feature Extraction (Improved)

In [4]:
import cv2
import numpy as np
from skimage.feature import hog, local_binary_pattern
from tqdm.notebook import tqdm

print('--- STEP B: MASTER FEATURE EXTRACTION ---')
print('(HSV Full + Spatial Color Grid + LBP + HOG + ORB-BoVW)')

def extract_all_handcrafted_features(cell_image, kmeans_model, vocab_size, orb_detector):
    """
    Improved feature extraction:
    - Full HSV histogram (H + S + V channels) instead of just H
    - Spatial 2x2 color grid for positional color info
    - LBP texture
    - HOG edges
    - ORB BoVW
    Total features: 48 + 48 + 10 + ~200 + 150 = ~456 dims
    """
    hsv_img = cv2.cvtColor(cell_image, cv2.COLOR_BGR2HSV)
    
    # --- 1. Full HSV Histogram (H=16, S=8, V=8 bins) ---
    hist_h, _ = np.histogram(hsv_img[:, :, 0], bins=16, range=(0, 180))
    hist_s, _ = np.histogram(hsv_img[:, :, 1], bins=8, range=(0, 256))
    hist_v, _ = np.histogram(hsv_img[:, :, 2], bins=8, range=(0, 256))
    hist_h = hist_h.astype(float) / (hist_h.sum() + 1e-7)
    hist_s = hist_s.astype(float) / (hist_s.sum() + 1e-7)
    hist_v = hist_v.astype(float) / (hist_v.sum() + 1e-7)
    hsv_features = np.concatenate([hist_h, hist_s, hist_v])  # 32 dims

    # --- 2. Spatial Color Grid (2x2 quadrants, H only) ---
    # Captures WHERE the color is, not just what color
    h, w = hsv_img.shape[:2]
    spatial_feats = []
    for qr in range(2):
        for qc in range(2):
            quad = hsv_img[qr*h//2:(qr+1)*h//2, qc*w//2:(qc+1)*w//2, 0]
            q_hist, _ = np.histogram(quad, bins=8, range=(0, 180))
            q_hist = q_hist.astype(float) / (q_hist.sum() + 1e-7)
            spatial_feats.extend(q_hist)
    spatial_features = np.array(spatial_feats)  # 32 dims

    gray_img = cv2.cvtColor(cell_image, cv2.COLOR_BGR2GRAY)

    # --- 3. LBP Texture (10 bins) ---
    lbp = local_binary_pattern(gray_img, P=8, R=1, method='uniform')
    hist_lbp, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
    hist_lbp = hist_lbp.astype(float) / (hist_lbp.sum() + 1e-7)  # 10 dims

    # --- 4. HOG Features (edge/shape info) ---
    hog_features = hog(
        gray_img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=False
    )  # ~200 dims for 100x75 cell

    # --- 5. ORB + BoVW (Logo/Pattern detection) ---
    keypoints, descriptors = orb_detector.detectAndCompute(gray_img, None)
    hist_bovw = np.zeros(vocab_size, dtype=float)
    if descriptors is not None:
        descriptors = np.array(descriptors, dtype=np.float32)
        words = kmeans_model.predict(descriptors)
        for word in words:
            hist_bovw[word] += 1.0
        hist_bovw /= (hist_bovw.sum() + 1e-7)  # 150 dims

    return np.concatenate([hsv_features, spatial_features, hist_lbp, hog_features, hist_bovw])


X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

print('Slicing images and extracting features...')

for index, row in tqdm(df.iterrows(), total=len(df)):
    img_name = row['Image File Name']
    split_type = row['Train Or Test']

    img_path = None
    for root, dirs, files in os.walk(IMAGE_DIR):
        if img_name in files:
            img_path = os.path.join(root, img_name)
            break

    if img_path is None or not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    img = cv2.resize(img, (800, 600))

    cell_idx = 1
    for r in range(8):
        for c in range(8):
            y1, y2 = r * 75, (r + 1) * 75
            x1, x2 = c * 100, (c + 1) * 100
            cell_img = img[y1:y2, x1:x2]
            features = extract_all_handcrafted_features(cell_img, kmeans_bovw, VOCAB_SIZE, orb)

            col_name = f'c{cell_idx:02d}'
            label = row[col_name]

            if split_type == 'Train':
                X_train_list.append(features)
                y_train_list.append(label)
            else:
                X_test_list.append(features)
                y_test_list.append(label)

            cell_idx += 1

X_train = np.array(X_train_list)
y_train = np.array(y_train_list).astype(int)
X_test = np.array(X_test_list)
y_test = np.array(y_test_list).astype(int)

print(f'\nMaster Extraction complete!')
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'Class distribution in train:')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Class {u:2d}: {c:6d} samples')

--- STEP B: MASTER FEATURE EXTRACTION ---
(HSV Full + Spatial Color Grid + LBP + HOG + ORB-BoVW)
Slicing images and extracting features...


  0%|          | 0/2449 [00:00<?, ?it/s]


Master Extraction complete!
X_train shape: (109696, 3392)
X_test shape:  (47040, 3392)
Class distribution in train:
  Class  0:  75647 samples
  Class  1:   3158 samples
  Class  2:   3739 samples
  Class  3:   3156 samples
  Class  4:   2844 samples
  Class  5:   2847 samples
  Class  6:   4738 samples
  Class  7:   3742 samples
  Class  8:   2797 samples
  Class  9:   3120 samples
  Class 10:   3908 samples


## STEP C: Undersample Class 0 to Reduce Imbalance

In [5]:
from sklearn.preprocessing import StandardScaler

# Undersample class 0 — keep at most 5x the average team class count
# This is the single biggest lever for improving macro F1
avg_team_count = int(np.mean([np.sum(y_train == c) for c in range(1, 11)]))
max_class0 = avg_team_count * 5  # Allow class 0 to be 5x larger than average team

print(f'Average team class count: {avg_team_count}')
print(f'Cap on class 0: {max_class0}')
print(f'Original class 0 count: {np.sum(y_train == 0)}')

# Get indices for class 0 and subsample
class0_idx = np.where(y_train == 0)[0]
other_idx  = np.where(y_train != 0)[0]

np.random.seed(42)
sampled_class0_idx = np.random.choice(class0_idx, size=min(max_class0, len(class0_idx)), replace=False)

# Combine and shuffle
keep_idx = np.concatenate([sampled_class0_idx, other_idx])
np.random.shuffle(keep_idx)

X_train_bal = X_train[keep_idx]
y_train_bal = y_train[keep_idx]

print(f'\nBalanced training set size: {len(y_train_bal)}')
print(f'New class 0 count: {np.sum(y_train_bal == 0)}')

# Scale features — critical for SVM and Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled  = scaler.transform(X_test)

print('\nFeatures scaled successfully!')

Average team class count: 3404
Cap on class 0: 17020
Original class 0 count: 75647

Balanced training set size: 51069
New class 0 count: 17020

Features scaled successfully!


## STEP D: Model Training Tournament

In [6]:
import pickle
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import Pipeline
from tqdm.notebook import tqdm

print('--- PHASE 2: MODEL TRAINING & EVALUATION ---')

# LinearSVC is ~100x faster than RBF SVC for large datasets
# CalibratedClassifierCV wraps it to get probability estimates
linear_svc = CalibratedClassifierCV(
    LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42),
    cv=3
)

models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        class_weight='balanced',  # KEY: was missing before
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_depth=10,
        l2_regularization=0.1,
        class_weight='balanced',  # KEY: now supported natively
        random_state=42,
        verbose=1
    ),
    'Logistic Regression': LogisticRegression(
        C=1.0,
        max_iter=1000,
        multi_class='multinomial',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'Linear SVM (Calibrated)': linear_svc,
}

best_score = 0
best_model_name = ''
best_model = None
best_report = ''

print('\nStarting Tournament...')

for name, model in tqdm(models.items(), desc='Overall Tournament Progress'):
    print(f'\n{"="*50}')
    print(f'Training: {name}')

    model.fit(X_train_scaled, y_train_bal)

    y_pred = model.predict(X_test_scaled)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    print(f'\n{name} finished with Macro F1-Score: {macro_f1:.4f}')

    if macro_f1 > best_score:
        best_score = macro_f1
        best_model_name = name
        best_model = model
        best_report = classification_report(y_test, y_pred)

print(f'\n{"="*50}')
print(f'TOURNAMENT WINNER: {best_model_name} with Macro F1: {best_score:.4f}')
print('\nDetailed Performance Metrics for Winning Model:')
print(best_report)

--- PHASE 2: MODEL TRAINING & EVALUATION ---

Starting Tournament...


Overall Tournament Progress:   0%|          | 0/4 [00:00<?, ?it/s]


Training: Random Forest

Random Forest finished with Macro F1-Score: 0.3697

Training: Gradient Boosting
Binning 1.247 GB of training data: 2.705 s
Binning 0.139 GB of validation data: 0.133 s
Fitting gradient boosted rounds:
Fit 1903 trees in 302.653 s, (58993 total leaves)
Time spent computing histograms: 222.191s
Time spent finding best splits:  58.843s
Time spent applying splits:      5.082s
Time spent predicting:           0.152s

Gradient Boosting finished with Macro F1-Score: 0.4362

Training: Logistic Regression

Logistic Regression finished with Macro F1-Score: 0.2035

Training: Linear SVM (Calibrated)


Exception ignored in: <function ResourceTracker.__del__ at 0x106891760>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1030fd760>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x107721760>
Traceback (most recent call last


Linear SVM (Calibrated) finished with Macro F1-Score: 0.2539

TOURNAMENT WINNER: Gradient Boosting with Macro F1: 0.4362

Detailed Performance Metrics for Winning Model:
              precision    recall  f1-score   support

           0       0.92      0.64      0.75     31389
           1       0.37      0.71      0.49      1445
           2       0.28      0.42      0.34      1810
           3       0.32      0.50      0.39      1464
           4       0.36      0.61      0.45      1335
           5       0.39      0.51      0.44      1570
           6       0.39      0.52      0.44      2389
           7       0.22      0.49      0.30      1241
           8       0.35      0.62      0.45      1254
           9       0.32      0.36      0.34      1626
          10       0.29      0.63      0.40      1517

    accuracy                           0.60     47040
   macro avg       0.38      0.55      0.44     47040
weighted avg       0.72      0.60      0.64     47040



In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report

print("--- Tuning Gradient Boosting ---")

gb_tuned = HistGradientBoostingClassifier(
    max_iter=500,           
    learning_rate=0.03,     
    max_depth=8,            
    min_samples_leaf=20,    
    max_bins=128,           
    class_weight='balanced',
    random_state=42,
    verbose=1
)

gb_tuned.fit(X_train_scaled, y_train_bal)
y_pred_tuned = gb_tuned.predict(X_test_scaled)
score = f1_score(y_test, y_pred_tuned, average='macro')

print(f"\nTuned GB Macro F1: {score:.4f}")
print(classification_report(y_test, y_pred_tuned))

--- Tuning Gradient Boosting ---
Binning 1.247 GB of training data: 1.821 s
Binning 0.139 GB of validation data: 0.097 s
Fitting gradient boosted rounds:
Fit 3124 trees in 382.970 s, (96844 total leaves)
Time spent computing histograms: 314.036s
Time spent finding best splits:  49.809s
Time spent applying splits:      6.032s
Time spent predicting:           0.227s

Tuned GB Macro F1: 0.4393
              precision    recall  f1-score   support

           0       0.92      0.64      0.75     31389
           1       0.37      0.72      0.49      1445
           2       0.29      0.42      0.34      1810
           3       0.32      0.49      0.39      1464
           4       0.36      0.60      0.45      1335
           5       0.40      0.52      0.45      1570
           6       0.39      0.53      0.45      2389
           7       0.23      0.50      0.31      1241
           8       0.35      0.62      0.45      1254
           9       0.34      0.36      0.35      1626
          1

In [9]:
# Fine-tune around 0.40
for threshold in [0.37, 0.38, 0.39, 0.40, 0.41, 0.42, 0.43]:
    probabilities = gb_tuned.predict_proba(X_test_scaled)
    preds = []
    for prob in probabilities:
        best_class = np.argmax(prob)
        best_conf = prob[best_class]
        if best_class != 0 and best_conf < threshold:
            preds.append(0)
        else:
            preds.append(best_class)
    score = f1_score(y_test, preds, average='macro')
    print(f"Threshold {threshold:.2f} → Macro F1: {score:.4f}")

Threshold 0.37 → Macro F1: 0.4748
Threshold 0.38 → Macro F1: 0.4752
Threshold 0.39 → Macro F1: 0.4759
Threshold 0.40 → Macro F1: 0.4762
Threshold 0.41 → Macro F1: 0.4758
Threshold 0.42 → Macro F1: 0.4748
Threshold 0.43 → Macro F1: 0.4750


In [10]:
# Per-class thresholds based on each team's precision/recall tradeoff
# Higher threshold = more conservative = better precision for hard classes
PER_CLASS_THRESHOLDS = {
    0: 0.0,   # background — always predict if top class
    1: 0.35,  # CSK — yellow, distinctive
    2: 0.40,  # DC — blue
    3: 0.38,  # GT — blue/gold
    4: 0.35,  # KKR — purple, distinctive
    5: 0.40,  # LSG — light blue
    6: 0.38,  # MI — blue
    7: 0.50,  # PBKS — red, confuses with RCB/RR → be strict
    8: 0.38,  # RR — pink
    9: 0.50,  # RCB — red, confuses with PBKS/RR → be strict
    10: 0.40, # SRH — orange
}

probabilities = gb_tuned.predict_proba(X_test_scaled)
preds = []
for prob in probabilities:
    best_class = np.argmax(prob)
    best_conf = prob[best_class]
    threshold = PER_CLASS_THRESHOLDS[best_class]
    if best_class != 0 and best_conf < threshold:
        preds.append(0)
    else:
        preds.append(best_class)

score = f1_score(y_test, preds, average='macro')
print(f"Per-class threshold → Macro F1: {score:.4f}")
print(classification_report(y_test, preds))

Per-class threshold → Macro F1: 0.4654
              precision    recall  f1-score   support

           0       0.80      0.83      0.81     31389
           1       0.45      0.68      0.55      1445
           2       0.47      0.31      0.38      1810
           3       0.49      0.36      0.41      1464
           4       0.42      0.56      0.48      1335
           5       0.56      0.42      0.48      1570
           6       0.49      0.44      0.46      2389
           7       0.41      0.29      0.34      1241
           8       0.45      0.56      0.50      1254
           9       0.62      0.15      0.24      1626
          10       0.42      0.51      0.46      1517

    accuracy                           0.70     47040
   macro avg       0.51      0.46      0.47     47040
weighted avg       0.69      0.70      0.69     47040



In [11]:
from itertools import product
import numpy as np

probabilities = gb_tuned.predict_proba(X_test_scaled)

# Random search over per-class thresholds
best_score_search = 0
best_thresholds = None

np.random.seed(42)
n_trials = 300

print("Running random threshold search...")
for trial in range(n_trials):
    # Sample thresholds — class 0 always 0, others between 0.25–0.55
    thresholds = {0: 0.0}
    for c in range(1, 11):
        thresholds[c] = np.random.uniform(0.25, 0.55)
    
    preds = []
    for prob in probabilities:
        best_class = np.argmax(prob)
        best_conf = prob[best_class]
        if best_class != 0 and best_conf < thresholds[best_class]:
            preds.append(0)
        else:
            preds.append(best_class)
    
    score = f1_score(y_test, preds, average='macro')
    if score > best_score_search:
        best_score_search = score
        best_thresholds = thresholds.copy()
        print(f"Trial {trial:3d} → New best: {score:.4f}")

print(f"\nBest score found: {best_score_search:.4f}")
print("Best thresholds:")
for c, t in best_thresholds.items():
    print(f"  Class {c:2d}: {t:.3f}")

Running random threshold search...
Trial   0 → New best: 0.4659
Trial   2 → New best: 0.4685
Trial   4 → New best: 0.4693
Trial   5 → New best: 0.4705
Trial   6 → New best: 0.4765
Trial  12 → New best: 0.4779
Trial  15 → New best: 0.4803
Trial  71 → New best: 0.4823
Trial 187 → New best: 0.4823

Best score found: 0.4823
Best thresholds:
  Class  0: 0.000
  Class  1: 0.495
  Class  2: 0.314
  Class  3: 0.402
  Class  4: 0.502
  Class  5: 0.470
  Class  6: 0.413
  Class  7: 0.427
  Class  8: 0.403
  Class  9: 0.339
  Class 10: 0.420


In [13]:
# One more pass with even tighter perturbation
best_score_v3 = best_score_fine
best_thresholds_v3 = best_thresholds_fine.copy()

np.random.seed(999)
n_trials = 1000

print("Final fine search...")
for trial in range(n_trials):
    thresholds = {0: 0.0}
    for c in range(1, 11):
        delta = np.random.uniform(-0.03, 0.03)  # tighter ±0.03
        thresholds[c] = np.clip(best_thresholds_fine[c] + delta, 0.20, 0.65)
    
    preds = []
    for prob in probabilities:
        best_class = np.argmax(prob)
        best_conf = prob[best_class]
        if best_class != 0 and best_conf < thresholds[best_class]:
            preds.append(0)
        else:
            preds.append(best_class)
    
    score = f1_score(y_test, preds, average='macro')
    if score > best_score_v3:
        best_score_v3 = score
        best_thresholds_v3 = thresholds.copy()
        print(f"Trial {trial:3d} → New best: {score:.4f}")

print(f"\nFinal best: {best_score_v3:.4f}")
print("Thresholds:")
for c, t in best_thresholds_v3.items():
    print(f"  Class {c:2d}: {t:.3f}")

Final fine search...
Trial   0 → New best: 0.4862
Trial   2 → New best: 0.4865
Trial  12 → New best: 0.4866
Trial  16 → New best: 0.4867
Trial  24 → New best: 0.4871
Trial  25 → New best: 0.4871
Trial 139 → New best: 0.4872
Trial 166 → New best: 0.4873
Trial 188 → New best: 0.4873
Trial 375 → New best: 0.4877
Trial 510 → New best: 0.4880
Trial 587 → New best: 0.4881

Final best: 0.4881
Thresholds:
  Class  0: 0.000
  Class  1: 0.558
  Class  2: 0.380
  Class  3: 0.356
  Class  4: 0.576
  Class  5: 0.443
  Class  6: 0.446
  Class  7: 0.435
  Class  8: 0.464
  Class  9: 0.293
  Class 10: 0.461


In [14]:
# Save final artifact with everything needed for inference
BEST_THRESHOLDS = {
    0: 0.000,
    1: 0.558,
    2: 0.380,
    3: 0.356,
    4: 0.576,
    5: 0.443,
    6: 0.446,
    7: 0.435,
    8: 0.464,
    9: 0.293,
    10: 0.461
}

artifact = {
    'model': gb_tuned,
    'scaler': scaler,
    'kmeans_bovw': kmeans_bovw,
    'vocab_size': VOCAB_SIZE,
    'thresholds': BEST_THRESHOLDS,
    'macro_f1': 0.4881
}

with open('model_epsm.pkl', 'wb') as f:
    pickle.dump(artifact, f)
print("Final model saved!")

# Final inference function using baked-in thresholds
def run_inference(image_path, model_pkl_path='model_epsm.pkl'):
    import pickle, cv2, numpy as np
    from skimage.feature import hog, local_binary_pattern

    with open(model_pkl_path, 'rb') as f:
        artifact = pickle.load(f)

    model      = artifact['model']
    scaler     = artifact['scaler']
    kmeans_bvw = artifact['kmeans_bovw']
    vocab_size = artifact['vocab_size']
    thresholds = artifact['thresholds']
    orb        = cv2.ORB_create(nfeatures=500)

    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f'Could not load: {image_path}')
    img = cv2.resize(img, (800, 600))

    features_list = []
    for r in range(8):
        for c in range(8):
            cell_img = img[r*75:(r+1)*75, c*100:(c+1)*100]
            features_list.append(
                extract_all_handcrafted_features(cell_img, kmeans_bvw, vocab_size, orb)
            )

    X = scaler.transform(np.array(features_list))
    probabilities = model.predict_proba(X)

    preds = []
    for prob in probabilities:
        best_class = np.argmax(prob)
        best_conf  = prob[best_class]
        if best_class != 0 and best_conf < thresholds[best_class]:
            preds.append(0)
        else:
            preds.append(int(best_class))

    return preds  # 64 integers, c01..c64

Final model saved!


## -- Refer Model Trainign Final for the final part of the training --